# 🤖 FAQ Chatbot — NLP Powered
**Topic:** Chatbot for FAQs using Cosine Similarity & NLTK

---

<h1>CELL 1 — Imports & NLTK Downloads</h1>

In [ ]:
import nltk
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print('✅ All libraries loaded successfully!')

✅ All libraries loaded successfully!


<h1>CELL 2 — FAQ Dataset (Smartphone Product)</h1>

In [ ]:
faqs = [
    # Pricing & Purchase
    {"question": "What is the price of the phone?",
     "answer": "💰 The PhoneX Pro starts at ₹49,999 for the base 128GB model and ₹59,999 for the 256GB variant."},

    {"question": "Where can I buy this phone?",
     "answer": "🛒 You can buy PhoneX Pro on our official website, Amazon, Flipkart, and authorized retail stores near you."},

    {"question": "Is EMI available for purchase?",
     "answer": "💳 Yes! EMI options are available starting from ₹2,083/month with 0% interest on select bank cards for 24 months."},

    # Battery
    {"question": "What is the battery capacity?",
     "answer": "🔋 PhoneX Pro packs a massive 5000mAh battery that lasts up to 2 days on normal usage."},

    {"question": "How long does it take to charge?",
     "answer": "⚡ With 65W fast charging, PhoneX Pro charges 0-100% in just 45 minutes!"},

    {"question": "Does the phone support wireless charging?",
     "answer": "📡 Yes, PhoneX Pro supports 15W wireless charging and 5W reverse wireless charging."},

    # Camera
    {"question": "What is the camera quality?",
     "answer": "📸 PhoneX Pro has a 108MP main sensor, 12MP ultra-wide, and 10MP periscope telephoto with 10x optical zoom."},

    {"question": "Can I record 4K videos?",
     "answer": "🎥 Absolutely! PhoneX Pro records 4K at 60fps and also supports 8K video at 24fps."},

    {"question": "What is the front camera resolution?",
     "answer": "🤳 The front camera is 32MP with autofocus and supports 4K selfie video recording."},

    # Display
    {"question": "What is the screen size?",
     "answer": "📱 PhoneX Pro features a 6.7-inch AMOLED display with a resolution of 2K (3200x1440 pixels)."},

    {"question": "What is the refresh rate?",
     "answer": "🔄 It has a 120Hz adaptive refresh rate display that adjusts between 1Hz and 120Hz to save battery."},

    {"question": "Is the display water resistant?",
     "answer": "💧 Yes! PhoneX Pro is IP68 rated, meaning it's waterproof up to 1.5m for 30 minutes."},

    # Performance
    {"question": "Which processor does it use?",
     "answer": "⚙️ PhoneX Pro is powered by the Snapdragon 8 Gen 3 processor — one of the fastest mobile chips available."},

    {"question": "How much RAM does the phone have?",
     "answer": "🧠 It comes with 12GB LPDDR5X RAM as standard, with a 16GB variant also available."},

    {"question": "Is the phone good for gaming?",
     "answer": "🎮 Yes! With Snapdragon 8 Gen 3, 120Hz display, and vapor cooling, PhoneX Pro handles even the most demanding games smoothly."},

    # Software & Features
    {"question": "What Android version does it run?",
     "answer": "🤖 PhoneX Pro ships with Android 15 and is guaranteed 4 years of OS updates and 5 years of security patches."},

    {"question": "Does it have a fingerprint sensor?",
     "answer": "👆 Yes, it has an under-display optical fingerprint sensor that unlocks the phone in 0.3 seconds."},

    {"question": "Does the phone support 5G?",
     "answer": "📶 Yes! PhoneX Pro is a 5G-enabled device supporting all major 5G bands in India."},

    # Warranty & Support
    {"question": "What is the warranty period?",
     "answer": "🛡️ PhoneX Pro comes with 1 year manufacturer warranty on device and 6 months on accessories."},

    {"question": "How do I contact customer support?",
     "answer": "📞 You can reach us at support@phonex.com, call 1800-XXX-XXXX (toll free), or visit our website's live chat 24/7."},

    {"question": "Can I return the phone if I don't like it?",
     "answer": "↩️ Yes! We offer a 7-day no-questions-asked return policy if purchased from our official website."},

    # Storage & Connectivity
    {"question": "Does the phone have expandable storage?",
     "answer": "💾 PhoneX Pro does not support a microSD card, but comes in 128GB, 256GB, and 512GB internal storage options."},

    {"question": "Does it have a headphone jack?",
     "answer": "🎧 No, PhoneX Pro uses USB-C for audio. However, a USB-C to 3.5mm adapter is included in the box."},

    {"question": "What colors are available?",
     "answer": "🎨 PhoneX Pro is available in Midnight Black, Pearl White, Aurora Blue, and Sunset Gold."}
]

print(f'✅ FAQ Dataset loaded! Total FAQs: {len(faqs)}')

✅ FAQ Dataset loaded! Total FAQs: 24


<h1>CELL 3 — NLP Preprocessing Engine</h1>

In [ ]:

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Words to KEEP even if stopwords (important for FAQ matching)
keep_words = {'not', 'no', 'how', 'what', 'when', 'where', 'which', 'who', 'why', 'can', 'does', 'is', 'are'}
stop_words = stop_words - keep_words

def preprocess(text):
    """Clean and normalize text using NLP techniques."""
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 3: Tokenize
    tokens = word_tokenize(text)
    
    # Step 4: Remove stopwords + Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    return ' '.join(tokens)

# Preprocess all FAQ questions
faq_questions = [faq['question'] for faq in faqs]
faq_answers   = [faq['answer']   for faq in faqs]
processed_questions = [preprocess(q) for q in faq_questions]

print('✅ NLP Preprocessing complete!')
print('\n📝 Example preprocessing:')
print(f'  Original : {faq_questions[0]}')
print(f'  Processed: {processed_questions[0]}')

✅ NLP Preprocessing complete!

📝 Example preprocessing:
  Original : What is the price of the phone?
  Processed: what is price phone


<h1>CELL 4 — TF-IDF Vectorizer + Cosine Similarity Engine</h1>

In [ ]:
# Build TF-IDF matrix from FAQ questions
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # Unigrams + Bigrams
    max_features=5000,     # Top 5000 features
    sublinear_tf=True      # Apply log normalization
)

tfidf_matrix = vectorizer.fit_transform(processed_questions)

def get_best_answer(user_query, threshold=0.15):
    """
    Match user query to most similar FAQ using Cosine Similarity.
    Returns (answer, confidence_score, matched_question)
    """
    # Preprocess user query
    processed_query = preprocess(user_query)
    
    if not processed_query.strip():
        return None, 0, None
    
    # Vectorize the user query
    query_vec = vectorizer.transform([processed_query])
    
    # Compute cosine similarity against all FAQs
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Get best match index
    best_idx = np.argmax(similarities)
    best_score = similarities[best_idx]
    
    if best_score >= threshold:
        return faq_answers[best_idx], round(float(best_score) * 100, 1), faq_questions[best_idx]
    else:
        return None, round(float(best_score) * 100, 1), None

# Quick test
test_q = "How much does the phone cost?"
ans, score, matched = get_best_answer(test_q)
print('✅ Cosine Similarity Engine ready!')
print(f'\n🧪 Test Query   : "{test_q}"')
print(f'   Matched FAQ  : "{matched}"')
print(f'   Confidence   : {score}%')
print(f'   Answer       : {ans}')

✅ Cosine Similarity Engine ready!

🧪 Test Query   : "How much does the phone cost?"
   Matched FAQ  : "How much RAM does the phone have?"
   Confidence   : 74.8%
   Answer       : 🧠 It comes with 12GB LPDDR5X RAM as standard, with a 16GB variant also available.


<h1>CELL 5 — Beautiful Chat UI with ipywidgets</h1>

In [ ]:
# ---------- Styles ----------
chat_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;600;700&display=swap');

.chat-wrapper {
    font-family: 'Space Grotesk', sans-serif;
    max-width: 700px;
    margin: 0 auto;
}
.chat-header {
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
    border-radius: 20px 20px 0 0;
    padding: 20px 24px;
    display: flex;
    align-items: center;
    gap: 14px;
}
.bot-avatar {
    width: 48px; height: 48px;
    background: linear-gradient(135deg, #f093fb, #f5576c);
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 22px;
}
.header-text h3 { margin:0; color:#fff; font-size:18px; font-weight:700; }
.header-text p  { margin:2px 0 0; color:#a8a8c8; font-size:12px; }
.online-dot {
    width:9px; height:9px; background:#00e676;
    border-radius:50%; display:inline-block; margin-right:5px;
    animation: pulse 1.5s infinite;
}
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }

.chat-body {
    background: #0d0d1a;
    min-height: 420px;
    padding: 20px;
    overflow-y: auto;
    border-left: 1px solid #1e1e3a;
    border-right: 1px solid #1e1e3a;
}
.msg-row { display:flex; margin-bottom:16px; align-items:flex-end; gap:10px; }
.msg-row.user { flex-direction:row-reverse; }

.bubble {
    max-width: 75%;
    padding: 12px 16px;
    border-radius: 18px;
    font-size: 14px;
    line-height: 1.6;
    word-break: break-word;
}
.bubble.bot {
    background: linear-gradient(135deg, #1e1e4a, #252550);
    color: #e0e0ff;
    border-bottom-left-radius: 4px;
    border: 1px solid #333366;
}
.bubble.user {
    background: linear-gradient(135deg, #f093fb, #f5576c);
    color: #fff;
    border-bottom-right-radius: 4px;
}
.bubble.error {
    background: linear-gradient(135deg, #2d1b1b, #3d1f1f);
    color: #ff8a80;
    border: 1px solid #5c2020;
}
.mini-avatar {
    width:28px; height:28px; border-radius:50%;
    display:flex; align-items:center; justify-content:center;
    font-size:14px; flex-shrink:0;
}
.mini-avatar.bot { background: linear-gradient(135deg, #f093fb, #f5576c); }
.mini-avatar.user { background: linear-gradient(135deg, #4facfe, #00f2fe); }

.confidence-tag {
    font-size:10px; color:#6666aa;
    margin-top:4px; padding-left:2px;
}
.matched-q {
    font-size:11px; color:#5555aa; font-style:italic;
    margin-top:3px;
}

.suggestion-chips {
    display:flex; flex-wrap:wrap; gap:8px;
    margin-top:10px;
}
.chip {
    background:#1a1a3a; color:#a0a0dd;
    border:1px solid #303060; border-radius:20px;
    padding:5px 12px; font-size:12px; cursor:pointer;
    transition:all 0.2s;
}
.chip:hover { background:#252560; color:#fff; border-color:#6060cc; }

.welcome-msg { text-align:center; padding:30px 20px; color:#4040aa; }
.welcome-msg .icon { font-size:48px; margin-bottom:12px; }
.welcome-msg h4 { color:#8080cc; margin:0 0 6px; font-size:16px; }
.welcome-msg p  { margin:0; font-size:13px; color:#404060; }
</style>
"""

# ---------- Suggested Questions ----------
SUGGESTIONS = [
    "What is the price?",
    "Battery backup?",
    "Camera specs?",
    "Does it have 5G?",
    "Warranty details?"
]

# ---------- State ----------
chat_history_html = []

def make_welcome():
    chips = ''.join([f'<span class="chip">{s}</span>' for s in SUGGESTIONS])
    return f"""
    <div class="welcome-msg">
        <div class="icon">🤖</div>
        <h4>Hi! I'm PhoneX Assistant</h4>
        <p>Ask me anything about PhoneX Pro!</p>
        <div class="suggestion-chips" style="justify-content:center; margin-top:14px;">{chips}</div>
    </div>
    """

def make_bot_bubble(text, score=None, matched=None):
    conf = f'<div class="confidence-tag">🎯 Confidence: {score}%</div>' if score else ''
    mq   = f'<div class="matched-q">📌 Matched: "{matched}"</div>' if matched else ''
    return f"""
    <div class="msg-row bot">
        <div class="mini-avatar bot">🤖</div>
        <div>
            <div class="bubble bot">{text}</div>
            {conf}{mq}
        </div>
    </div>
    """

def make_user_bubble(text):
    return f"""
    <div class="msg-row user">
        <div class="mini-avatar user">😊</div>
        <div class="bubble user">{text}</div>
    </div>
    """

def make_error_bubble(score):
    return f"""
    <div class="msg-row bot">
        <div class="mini-avatar bot">🤖</div>
        <div>
            <div class="bubble error">😕 Sorry, I couldn't find a good match for your question.<br>Try rephrasing it or ask about: price, battery, camera, display, storage, warranty etc.</div>
            <div class="confidence-tag">🎯 Best match score: {score}% (too low)</div>
        </div>
    </div>
    """

# ---------- Widgets ----------
output_area = widgets.Output()
text_input  = widgets.Text(
    placeholder='Type your question here...',
    layout=widgets.Layout(width='85%', height='40px')
)
send_btn = widgets.Button(
    description='Send ➤',
    button_style='',
    layout=widgets.Layout(width='13%', height='40px')
)
send_btn.style.button_color = '#f5576c'

def render_chat():
    """Render full chat window."""
    messages = ''.join(chat_history_html) if chat_history_html else make_welcome()
    html = f"""
    {chat_css}
    <div class="chat-wrapper">
        <div class="chat-header">
            <div class="bot-avatar">🤖</div>
            <div class="header-text">
                <h3>PhoneX Assistant</h3>
                <p><span class="online-dot"></span>Online • NLP Powered</p>
            </div>
        </div>
        <div class="chat-body">{messages}</div>
    </div>
    """
    with output_area:
        clear_output(wait=True)
        display(HTML(html))

def handle_send(btn_or_text):
    user_text = text_input.value.strip()
    if not user_text:
        return
    text_input.value = ''

    # Add user bubble
    chat_history_html.append(make_user_bubble(user_text))

    # Get answer
    answer, score, matched = get_best_answer(user_text)

    if answer:
        chat_history_html.append(make_bot_bubble(answer, score, matched))
    else:
        chat_history_html.append(make_error_bubble(score))

    render_chat()

def handle_enter(change):
    if change['new'].endswith('\n') or len(change['new']) > 0:
        pass  # handled by send button

send_btn.on_click(handle_send)
text_input.on_submit(handle_send)  # Press Enter to send

# ---------- Layout ----------
input_row = widgets.HBox(
    [text_input, send_btn],
    layout=widgets.Layout(
        width='700px',
        padding='10px',
        background_color='#0d0d1a',
        border='1px solid #1e1e3a',
        border_radius='0 0 20px 20px'
    )
)

chatbot_ui = widgets.VBox(
    [output_area, input_row],
    layout=widgets.Layout(max_width='700px')
)

render_chat()
display(chatbot_ui)

<h1>CELL 6 — (OPTIONAL) Test Similarity Scores — No UI needed</h1>

In [ ]:
test_queries = [
    "How much does the phone cost?",
    "Tell me about the battery life",
    "Is it waterproof?",
    "Which processor?",
    "Can I use it for gaming?",
    "How to return the product?",
    "What colours does it come in?",
    "Is there a headphone jack?",
    "How do I contact support?",
    "Does it support wireless charging?"
]

print('=' * 70)
print('  📊 COSINE SIMILARITY TEST RESULTS')
print('=' * 70)

for q in test_queries:
    ans, score, matched = get_best_answer(q)
    status = '✅' if ans else '❌'
    print(f'\n{status} Query   : {q}')
    print(f'   Matched : {matched}')
    print(f'   Score   : {score}%')
    print(f'   Answer  : {ans[:70] if ans else "No match found"}...')

print('\n' + '=' * 70)

  📊 COSINE SIMILARITY TEST RESULTS

✅ Query   : How much does the phone cost?
   Matched : How much RAM does the phone have?
   Score   : 74.8%
   Answer  : 🧠 It comes with 12GB LPDDR5X RAM as standard, with a 16GB variant also...

✅ Query   : Tell me about the battery life
   Matched : What is the battery capacity?
   Score   : 45.0%
   Answer  : 🔋 PhoneX Pro packs a massive 5000mAh battery that lasts up to 2 days o...

✅ Query   : Is it waterproof?
   Matched : What is the price of the phone?
   Score   : 25.0%
   Answer  : 💰 The PhoneX Pro starts at ₹49,999 for the base 128GB model and ₹59,99...

✅ Query   : Which processor?
   Matched : Which processor does it use?
   Score   : 69.0%
   Answer  : ⚙️ PhoneX Pro is powered by the Snapdragon 8 Gen 3 processor — one of ...

✅ Query   : Can I use it for gaming?
   Matched : Is the phone good for gaming?
   Score   : 26.0%
   Answer  : 🎮 Yes! With Snapdragon 8 Gen 3, 120Hz display, and vapor cooling, Phon...

✅ Query   : How to return th

---
## 🎓 How It Works

| Step | Technique | Library |
|------|-----------|--------|
| Text Cleaning | Regex + Lowercase | `re` |
| Tokenization | `word_tokenize` | `NLTK` |
| Stopword Removal | Custom filtered stopwords | `NLTK` |
| Lemmatization | `WordNetLemmatizer` | `NLTK` |
| Vectorization | TF-IDF (Bigrams) | `scikit-learn` |
| Similarity | Cosine Similarity | `scikit-learn` |
| UI | ipywidgets | `ipywidgets` |

---